In [1]:
%load_ext autoreload
%autoreload 2

from paper_utils import *
    
import os
# os.chdir('../..')
from sklearn.metrics import cohen_kappa_score


# Print entailment inputs for human rating

In [ ]:
runs = {
    '1r6kekju': 'squad-deberta',
    'f8yk94fy': 'trivia_qa-deberta',
    'lzlhybwt': 'bioasq-deberta',
}

configs = {}
for wandb_id in runs:
    configs[wandb_id] = restore_file(wandb_id, filenames=['config.yaml'])[0]
    
check_first_item(configs)

```
contradiction -> 0
neutral -> 1
entailment -> 2
```

In [286]:
def print4human(wandbid, MAX_NO=100, SKIP_UNTIL=0):

    name = runs[wandbid]
    model = '-'.join(name.split('-')[1:])
    slurmid = api.run(f'goatml/semantic_uncertainty/{wandbid}').notes.split(': ')[1]
    last_question = ''
    entailment_counter = 0
    entailment_preds = {}
    skip = False
    
    
    with open(f'../../log/slurm-{slurmid}.out', 'r') as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if 'INFO' in line:
                line = line[line.index('INFO') + len('INFO') + 4:]
    
            if f'{model} input:'.lower() in line.lower():
    
                original_line = line
    
                if model == 'deberta':
                    if 'weight room' in line:
                        line = 'How many weight rooms are in the Malkin Athletic Center'
                    elif '?' in line:
                        line = line[:line.index('?')]
                    elif '.' in line[:50]:
                        line = line[:line.index('.')]
                    else:
                        line = line[:40]
                
                if line == last_question:
                    skip = True
                    continue
                else:
                    print(entailment_counter, original_line)
                    skip = False
    
                last_question = line
                entailment_counter += 1
            
            sys.stdout.flush()
            if entailment_counter == MAX_NO:
                break


In [283]:
# squad
print4human('1r6kekju')

0  Deberta Input: What was Warsaw's population in 1901? According to the census of 1901, Warsaw's population was 756,475. -> What was Warsaw's population in 1901? According to the census of 1901, Warsaw's population was 817,000.

1  Deberta Input: When did O2 begin to acculturate in the atmosphere? O2 began to accumulate in the atmosphere approximately 2.7 billion years ago during the Great Oxygenation Event. -> When did O2 begin to acculturate in the atmosphere? Oxygen began to accumulate in the atmosphere around 2.7 billion years ago during the Great Oxygenation Event.

2  Deberta Input: Who was Frédéric Chopin? Frédéric Chopin was a Polish-French Romantic composer and pianist known for his delicate, expressive, and technically demanding piano music. -> Who was Frédéric Chopin? Frédéric Chopin was a Polish composer and pianist of the Romantic era who is widely regarded as one of the greatest composers of all time, known for his technically demanding and expressive piano music, includ

In [287]:
# trivia-qa
print4human('f8yk94fy')

0  Deberta Input: Nigel Hawthorne was Oscar nominated for The Madness of which King? Nigel Hawthorne was Oscar nominated for The Madness of King George. -> Nigel Hawthorne was Oscar nominated for The Madness of which King? Nigel Hawthorne was Oscar-nominated for his portrayal of King George III in The Madness of King George.

1  Deberta Input: Which actress and singer's biography was entitled 'The Other Side Of The Rainbow'? Judy Garland's biography was entitled 'The Other Side Of The Rainbow'. -> Which actress and singer's biography was entitled 'The Other Side Of The Rainbow'? The actress and singer whose biography was entitled 'The Other Side Of The Rainbow' was Judy Garland.

2  Deberta Input: The actor John Wayne was known by what nickname? John Wayne was known by the nickname "The Duke." -> The actor John Wayne was known by what nickname? The actor John Wayne was known by the nickname "The Duke."

3  Deberta Input: In Greek mythology, who did flute playing shepherd Marsyas challe

In [288]:
# bioasq
print4human('lzlhybwt')

0  Deberta Input: Computational tools for predicting allosteric pathways in proteins Computational tools for predicting allosteric pathways in proteins include molecular dynamics simulations, molecular docking, machine learning algorithms, and structure-based design methods. -> Computational tools for predicting allosteric pathways in proteins Computational tools for predicting allosteric pathways in proteins include molecular dynamics simulations, structure-based modeling, machine learning algorithms, and graph theory-based methods, which can aid in identifying potential allosteric sites, predicting the binding of small molecules or protein ligands, and understanding the underlying mechanisms of allostery.

1  Deberta Input: Which is the molecular mechanism underlying K-ras alterations in carcinomas? K-ras alterations in carcinomas involve mutations in the KRAS gene, which can result in the production of a constitutively active K-ras protein that promotes cell proliferation and surviv

# Extract Entailment for existing runs from logs

In [381]:
runs = {
    # '8ehbkd99': 'squad-gpt-4',
    # '5bqecpaq': 'squad-gpt-3.5',
    # 'eriayh79': 'squad-llama-2-70b-chat',
    # '1r6kekju': 'squad-deberta',

    # '37it7nr9': 'trivia_qa-gpt-4',
    # 'mpgj9mz3': 'trivia_qa-gpt-3.5',
    # 'qfwl6vze': 'trivia_qa-llama-2-70b-chat',
    # 'f8yk94fy': 'trivia_qa-deberta',
    
    '3d6obogq': 'bioasq-gpt-4',
    'jj5da8bd': 'bioasq-gpt-3.5',
    '8i0520i0': 'bioasq-llama-2-70b-chat',
    'lzlhybwt': 'bioasq-deberta',
}

In [378]:
{v: get_slurmid(k) for k, v in runs.items()}

{'bioasq-gpt4': '167372',
 'bioasq-gpt3.5': '168316',
 'bioasq-deberta': '171335',
 'bioasq-llama-2-70b-chat': '171334'}

In [379]:
configs = {}
for wandb_id in runs:
    configs[wandb_id] = restore_file(wandb_id, filenames=['config.yaml'])[0]

get_slurmid = lambda wandbid: api.run(f'goatml/semantic_uncertainty/{wandbid}').notes.split(': ')[1]
check_first_item(configs)

3d6obogq 167372 bioasq  Computational tools for predicting allosteric pathways in proteins
jj5da8bd 168316 bioasq  Computational tools for predicting allosteric pathways in proteins
lzlhybwt 171335 bioasq  Computational tools for predicting allosteric pathways in proteins
8i0520i0 171334 bioasq  Computational tools for predicting allosteric pathways in proteins


In [385]:
# First check runs individually! Then check if they match up!
# Individually: check all 100 inputs make sense!

# CHECKING: does first item line up?
# SKIP_UNTIL = 0
# MAX_NO = 1
# verbose = True

# SKIP_UNTIL = 0
# MAX_NO = 10
# verbose = True

# SKIP_UNTIL = 3
# MAX_NO = 4
# verbose = True

# SKIP_UNTIL = 99
# MAX_NO = 100
# verbose = True

# MAX_NO = 51
# SKIP_UNTIL = 50
# verbose = True


# CHECKING: do things continue to line up at the very end?
# MAX_NO = 100
# SKIP_UNTIL = 99
# verbose = True

MAX_NO = 100
SKIP_UNTIL = 0
verbose = False

# verbose = False


def get_pred(line):
    if 'neutral' in line:
        return 1
    elif 'entailment' in line:
        return 2
    elif 'contradiction' in line:
        return 0
    elif 'prediction: 0' in line:
        return 0
    elif 'prediction: 1' in line:
        return 1
    elif 'prediction: 2' in line:
        return 2
    else:
        raise

preds = {}
for wandbid, name in runs.items():
    # if wandbid != 'f8yk94fy':
        # continue
    
    print(20 * 'xxx')
    print(wandbid, name)

    model = '-'.join(name.split('-')[1:])
    slurmid = get_slurmid(wandbid)
    last_question = ''
    entailment_counter = 0
    entailment_preds = {}
    skip = False
    hit = False
    with open(f'../../log/slurm-{slurmid}.out', 'r') as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            line = line.lower()

            if 'info' in line:
                line = line[line.index('info'):]

            if f'{model} input:' in line:
                original_line = line

                if model == 'deberta':
                    if 'weight room' in line:
                        line = 'How many weight rooms are in the Malkin Athletic Center'
                    elif '?' in line:
                        line = line[:line.index('?')]
                    elif '.' in line[:50]:
                        line = line[:line.index('.')]
                    else:
                        line = line[:40]
                
                if line == last_question:
                    skip = True
                    hit = False
                    continue
                else:
                    skip = False

                last_question = line
                if verbose and (entailment_counter >= SKIP_UNTIL):
                    print('----')
                    if model == 'deberta':
                        print(i, 'INPT', original_line)
                    else:
                        print(i, 'INPT', line)
                        print(i+2, 'INPT', lines[i+2])
                        print(i+3, 'INPT', lines[i+3])
                entailment_counter += 1

            if not skip and (f'{model} prediction:' in line):
                hit = True
                entailment_prediction = get_pred(line)
                if verbose and (entailment_counter > SKIP_UNTIL):
                    print(i, 'ANSWER', line)
                    print(i, 'ANSWER', entailment_prediction)
                entailment_preds[entailment_counter] = entailment_prediction
            
            sys.stdout.flush()
            if (entailment_counter == MAX_NO) and hit:
                break
    preds[wandbid] = entailment_preds

xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
3d6obogq bioasq-gpt-4
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
jj5da8bd bioasq-gpt-3.5
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
8i0520i0 bioasq-llama-2-70b-chat
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
lzlhybwt bioasq-deberta


In [386]:
df = pd.DataFrame.from_dict(preds).rename(columns=lambda x: runs[x])
# for rater in ['squad-human_jansen', 'squad-human_sebhar']:
for rater in ['bioasq-human_jansen']:
    df[rater] = 'FILL_IN'
df


,bioasq-gpt-4,bioasq-gpt-3.5,bioasq-llama-2-70b-chat,bioasq-deberta,bioasq-human_jansen
1,1,1,2,1,FILL_IN
2,1,2,2,1,FILL_IN
3,0,0,2,0,FILL_IN
4,2,2,2,1,FILL_IN
5,2,2,2,1,FILL_IN
...,...,...,...,...,...
96,0,2,0,1,FILL_IN
97,2,2,2,1,FILL_IN
98,2,1,2,2,FILL_IN
99,0,0,2,0,FILL_IN


In [387]:
tmp = df.reset_index().melt(id_vars='index', value_vars=df.columns)
tmp['dataset'] = tmp.variable.map(lambda x: x.split('-')[0])
tmp['model'] = tmp.variable.map(lambda x: '-'.join(x.split('-')[1:]))
tmp = tmp[['index', 'dataset', 'model', 'value']]
tmp['index'] = tmp['index'] - 1
tmp
# tmp.to_csv('23-11-24-entailment-evaluation.csv', index=False)
print(tmp.to_csv(index=False))

index,dataset,model,value
0,bioasq,gpt-4,1
1,bioasq,gpt-4,1
2,bioasq,gpt-4,0
3,bioasq,gpt-4,2
4,bioasq,gpt-4,2
5,bioasq,gpt-4,2
6,bioasq,gpt-4,0
7,bioasq,gpt-4,2
8,bioasq,gpt-4,2
9,bioasq,gpt-4,1
10,bioasq,gpt-4,1
11,bioasq,gpt-4,1
12,bioasq,gpt-4,2
13,bioasq,gpt-4,1
14,bioasq,gpt-4,2
15,bioasq,gpt-4,0
16,bioasq,gpt-4,1
17,bioasq,gpt-4,2
18,bioasq,gpt-4,1
19,bioasq,gpt-4,2
20,bioasq,gpt-4,2
21,bioasq,gpt-4,1
22,bioasq,gpt-4,2
23,bioasq,gpt-4,2
24,bioasq,gpt-4,2
25,bioasq,gpt-4,2
26,bioasq,gpt-4,2
27,bioasq,gpt-4,2
28,bioasq,gpt-4,2
29,bioasq,gpt-4,0
30,bioasq,gpt-4,0
31,bioasq,gpt-4,2
32,bioasq,gpt-4,2
33,bioasq,gpt-4,2
34,bioasq,gpt-4,2
35,bioasq,gpt-4,2
36,bioasq,gpt-4,0
37,bioasq,gpt-4,2
38,bioasq,gpt-4,1
39,bioasq,gpt-4,2
40,bioasq,gpt-4,2
41,bioasq,gpt-4,2
42,bioasq,gpt-4,1
43,bioasq,gpt-4,2
44,bioasq,gpt-4,2
45,bioasq,gpt-4,2
46,bioasq,gpt-4,2
47,bioasq,gpt-4,0
48,bioasq,gpt-4,2
49,bioasq,gpt-4,2
50,bioasq,gpt-4,2
51,bioasq,gpt-4,0
52,bioasq,gpt-4,2
53,bioasq,gpt-4,2
54,bioasq,gp

# Analysis: Correlation between entailment metrics and human judgement

In [462]:
df = pd.read_csv('23-11-28-entailment-evaluation.csv', index_col=None)

In [463]:
def remove_incomplete(df):
    # filter out incomplete data!
    ignore = []
    for metric, mdf in df.groupby('model'):
        if (mdf.value == 'FILL_IN').any():
            ignore.append(metric)
    print(f'Ignoring metrics {ignore} for now.')
    df = df[df.model.map(lambda x: x not in ignore)]
    return df

for name, tmp in df.groupby(['index', 'dataset', 'model']):
    if len(tmp) > 1:
        print(name)
        display(tmp)
    
df = remove_incomplete(df)
# df = df[df.value != 'FILL_IN']

# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    # display(df[df.dataset == 'trivia_qa'].pivot(index='index', columns='model', values='value').iloc[:100])

Ignoring metrics [] for now.


In [464]:
def style(df):
    return df.style.background_gradient(sns.color_palette('crest', as_cmap=True), axis=None).format(precision=2)

In [502]:
def agreement(x, y):
    return np.mean(x == y)


for method in [cohen_kappa_score, agreement, 'pearson', 'kendall', 'spearman']:
    print(90*'x')
    print(colorize(f'METHOD: {method}'))
    print(90*'x')

    corrs = []
    for dataset, gdf in df.groupby('dataset'):
        pdf = gdf.pivot(index='index', columns='model', values='value')
        corr = pdf.corr(method=method)
        corrs.append(corr)
        print(colorize(f'method: {method} -- dataset: {dataset}', 1))
        display(style(corr))

    avg = pd.concat(corrs).reset_index().groupby('model').mean()
    # print(colorize(f'METHOD: {method} --average_over_datasets', 2))
    # display(style(avg))

    tmp = pd.DataFrame([0.5 * (avg.loc['human_jansen'] + avg.loc['human_sebhar'])])
    tmp.index = ['human_average']
    avg = avg._append(tmp)
    print(colorize(f'METHOD: {method} --average_over_datasets', 2))
    display(style(avg))
    

xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: <function cohen_kappa_score at 0x7fed6421d940>
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
method: <function cohen_kappa_score at 0x7fed6421d940> -- dataset: bioasq


model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
model,,,,,,
deberta,1.00,0.24,0.45,0.51,0.40,0.19
gpt-3.5,0.24,1.00,0.54,0.38,0.37,0.42
gpt-4,0.45,0.54,1.00,0.65,0.52,0.41
human_jansen,0.51,0.38,0.65,1.00,0.54,0.33
human_sebhar,0.40,0.37,0.52,0.54,1.00,0.28
llama-2-70b-chat,0.19,0.42,0.41,0.33,0.28,1.00


method: <function cohen_kappa_score at 0x7fed6421d940> -- dataset: squad


model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
model,,,,,,
deberta,1.00,0.38,0.53,0.48,0.39,0.25
gpt-3.5,0.38,1.00,0.61,0.61,0.35,0.39
gpt-4,0.53,0.61,1.00,0.64,0.31,0.32
human_jansen,0.48,0.61,0.64,1.00,0.43,0.33
human_sebhar,0.39,0.35,0.31,0.43,1.00,0.35
llama-2-70b-chat,0.25,0.39,0.32,0.33,0.35,1.00


method: <function cohen_kappa_score at 0x7fed6421d940> -- dataset: trivia_qa


...

model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
model,,,,,,
deberta,1.00,0.71,0.75,0.63,0.63,0.52
gpt-3.5,0.71,1.00,0.76,0.70,0.68,0.57
gpt-4,0.75,0.76,1.00,0.76,0.65,0.49
human_jansen,0.63,0.70,0.76,1.00,0.69,0.50
human_sebhar,0.63,0.68,0.65,0.69,1.00,0.63
llama-2-70b-chat,0.52,0.57,0.49,0.50,0.63,1.00


method: spearman -- dataset: trivia_qa


model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
model,,,,,,
deberta,1.00,0.51,0.75,0.76,0.56,0.60
gpt-3.5,0.51,1.00,0.66,0.73,0.72,0.66
gpt-4,0.75,0.66,1.00,0.78,0.70,0.62
human_jansen,0.76,0.73,0.78,1.00,0.76,0.70
human_sebhar,0.56,0.72,0.70,0.76,1.00,0.65
llama-2-70b-chat,0.60,0.66,0.62,0.70,0.65,1.00


METHOD: spearman --average_over_datasets


model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
deberta,1.00,0.59,0.74,0.70,0.62,0.53
gpt-3.5,0.59,1.00,0.69,0.65,0.65,0.58
gpt-4,0.74,0.69,1.00,0.77,0.68,0.58
human_jansen,0.70,0.65,0.77,1.00,0.74,0.58
human_sebhar,0.62,0.65,0.68,0.74,1.00,0.60
llama-2-70b-chat,0.53,0.58,0.58,0.58,0.60,1.00
human_average,0.66,0.65,0.73,0.87,0.87,0.59


In [ ]:
# tmp = df.set_index('dataset').loc['trivia_qa'].set_index('model')
x, y = tmp.loc['human_sebhar'].value.values, tmp.loc['gpt-4'].value.values
print(np.mean(x == y))
print(cohen_kappa_score(x, y))

# I think it's kind of wild that an 87% agreement means the cohen kappa score is only 0.48?!

tmp = df.set_index('dataset').loc['squad'].set_index('model')
x, y = tmp.loc['human_jansen'].value.values, tmp.loc['gpt-4'].value.values
print(np.mean(x == y))
print(cohen_kappa_score(x, y))


# But I've got only 77% agreement and my cohen cappa score is 64
# ---> Maybe cohen cappa is not correct for us because the entailment scores 
# are not ordered?


# score might also be so small for trivia_qa because there is a large class imbalance..

**Cohens kappa is not the right metric for nominal data!!I.e. there is no order between the entailment scores.. (arguably?)**

https://john-uebersax.com/stat/agree.htm says the following for nominal data

* Assess raw agreement, overall and specific to each category.
* Often (perhaps usually), disregard the actual magnitude of kappa here; it is problematic with nominal data because ordinarily one can neither assume that all types of disagreement are equally serious (unweighted kappa) nor choose an objective set of differential disagreement weights (weighted kappa). If, however, it is genuinely true that all pairs of rating categories are equally "disparate", then the magnitude of Cohen's unweighted kappa can be interpreted as a form of intraclass correlation.


**actually, for strict entailment, we only care about [entailment, [neutral, contradiction]]. Maybe if we lump neutral and contradiction together, metrics improve?**

Okay, this helps somewhat (see below!) Now at least seb agrees with the gpts a bit more often!

In [501]:
ldf = df.copy()
ldf['value'] = ldf.value.map(lambda x: {0: 1}.get(x, x))

for method in [cohen_kappa_score, agreement, 'pearson', 'kendall', 'spearman']:
    print(90*'x')
    print(colorize(f'METHOD: {method}'))
    print(90*'x')

    corrs = []
    for dataset, gdf in ldf.groupby('dataset'):
        pdf = gdf.pivot(index='index', columns='model', values='value')
        corr = pdf.corr(method=method)
        corrs.append(corr)
        print(colorize(f'method: {method} -- dataset: {dataset}', 1))
        display(style(corr))

    avg = pd.concat(corrs).reset_index().groupby('model').mean()
    # print(colorize(f'METHOD: {method} --average_over_datasets', 2))
    # display(style(avg))

    tmp = pd.DataFrame([0.5 * (avg.loc['human_jansen'] + avg.loc['human_sebhar'])])
    tmp.index = ['human_average']
    avg = avg._append(tmp)
    print(colorize(f'METHOD: {method} --average_over_datasets', 2))
    display(style(avg))



xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: <function cohen_kappa_score at 0x7fed6421d940>
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
method: <function cohen_kappa_score at 0x7fed6421d940> -- dataset: bioasq


model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
model,,,,,,
deberta,1.00,0.27,0.53,0.56,0.45,0.17
gpt-3.5,0.27,1.00,0.57,0.36,0.48,0.42
gpt-4,0.53,0.57,1.00,0.67,0.62,0.42
human_jansen,0.56,0.36,0.67,1.00,0.67,0.30
human_sebhar,0.45,0.48,0.62,0.67,1.00,0.33
llama-2-70b-chat,0.17,0.42,0.42,0.30,0.33,1.00


method: <function cohen_kappa_score at 0x7fed6421d940> -- dataset: squad


model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
model,,,,,,
deberta,1.00,0.55,0.71,0.52,0.53,0.30
gpt-3.5,0.55,1.00,0.74,0.70,0.60,0.54
gpt-4,0.71,0.74,1.00,0.76,0.62,0.45
human_jansen,0.52,0.70,0.76,1.00,0.66,0.47
human_sebhar,0.53,0.60,0.62,0.66,1.00,0.49
llama-2-70b-chat,0.30,0.54,0.45,0.47,0.49,1.00


method: <function cohen_kappa_score at 0x7fed6421d940> -- dataset: trivia_qa


...

model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
model,,,,,,
deberta,1.00,0.61,0.73,0.55,0.56,0.42
gpt-3.5,0.61,1.00,0.75,0.70,0.61,0.60
gpt-4,0.73,0.75,1.00,0.76,0.62,0.54
human_jansen,0.55,0.70,0.76,1.00,0.66,0.54
human_sebhar,0.56,0.61,0.62,0.66,1.00,0.57
llama-2-70b-chat,0.42,0.60,0.54,0.54,0.57,1.00


method: spearman -- dataset: trivia_qa


model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
model,,,,,,
deberta,1.00,0.47,0.72,0.72,0.51,0.52
gpt-3.5,0.47,1.00,0.65,0.70,0.72,0.64
gpt-4,0.72,0.65,1.00,0.78,0.69,0.61
human_jansen,0.72,0.70,0.78,1.00,0.75,0.65
human_sebhar,0.51,0.72,0.69,0.75,1.00,0.66
llama-2-70b-chat,0.52,0.64,0.61,0.65,0.66,1.00


METHOD: spearman --average_over_datasets


model,deberta,gpt-3.5,gpt-4,human_jansen,human_sebhar,llama-2-70b-chat
deberta,1.00,0.47,0.69,0.63,0.53,0.41
gpt-3.5,0.47,1.00,0.66,0.59,0.61,0.57
gpt-4,0.69,0.66,1.00,0.74,0.65,0.55
human_jansen,0.63,0.59,0.74,1.00,0.69,0.52
human_sebhar,0.53,0.61,0.65,0.69,1.00,0.54
llama-2-70b-chat,0.41,0.57,0.55,0.52,0.54,1.00
human_average,0.58,0.60,0.69,0.85,0.85,0.53


alternatively, for weak entailment, we mostly care about [[entailment, neutral], [contradiction]]

In [461]:
ldf = df.copy()
ldf['value'] = ldf.value.map(lambda x: {1: 2}.get(x, x))

for method in [cohen_kappa_score, agreement, 'pearson', 'kendall', 'spearman']:
    print(90*'x')
    print(colorize(f'METHOD: {method}'))
    print(90*'x')

    corrs = []
    for dataset, gdf in ldf.groupby('dataset'):
        pdf = gdf.pivot(index='index', columns='model', values='value')
        corr = pdf.corr(method=method)
        corrs.append(corr)
        # print(colorize(f'method: {method} -- dataset: {dataset}', 1))
        # display(style(corr))

    avg = pd.concat(corrs).reset_index().groupby('model').mean()
    print(colorize(f'METHOD: {method} --average_over_datasets', 2))
    display(style(avg))
    

xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: <function cohen_kappa_score at 0x7fed6421d940>
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: <function cohen_kappa_score at 0x7fed6421d940> --average_over_datasets


model,deberta,gpt-3.5,gpt-4,human_sebhar,llama-2-70b-chat,human_jansen
model,,,,,,
deberta,1.00,0.54,0.56,0.60,0.57,0.56
gpt-3.5,0.54,1.00,0.60,0.40,0.55,0.57
gpt-4,0.56,0.60,1.00,0.47,0.49,0.58
human_jansen,0.56,0.57,0.58,0.45,0.25,1.00
human_sebhar,0.60,0.40,0.47,1.00,0.49,0.45
llama-2-70b-chat,0.57,0.55,0.49,0.49,1.00,0.25


xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: <function agreement at 0x7fed5848e980>
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: <function agreement at 0x7fed5848e980> --average_over_datasets


model,deberta,gpt-3.5,gpt-4,human_sebhar,llama-2-70b-chat,human_jansen
model,,,,,,
deberta,1.00,0.88,0.88,0.87,0.88,0.82
gpt-3.5,0.88,1.00,0.91,0.82,0.89,0.85
gpt-4,0.88,0.91,1.00,0.82,0.89,0.86
human_jansen,0.82,0.85,0.86,0.74,0.75,1.00
human_sebhar,0.87,0.82,0.82,1.00,0.84,0.74
llama-2-70b-chat,0.88,0.89,0.89,0.84,1.00,0.75


xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: pearson
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: pearson --average_over_datasets


model,deberta,gpt-3.5,gpt-4,human_sebhar,llama-2-70b-chat,human_jansen
model,,,,,,
deberta,1.00,0.57,0.58,0.61,0.59,0.57
gpt-3.5,0.57,1.00,0.61,0.44,0.56,0.58
gpt-4,0.58,0.61,1.00,0.50,0.51,0.61
human_jansen,0.57,0.58,0.61,0.48,0.26,1.00
human_sebhar,0.61,0.44,0.50,1.00,0.55,0.48
llama-2-70b-chat,0.59,0.56,0.51,0.55,1.00,0.26


xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: kendall
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: kendall --average_over_datasets


model,deberta,gpt-3.5,gpt-4,human_sebhar,llama-2-70b-chat,human_jansen
model,,,,,,
deberta,1.00,0.57,0.58,0.61,0.59,0.57
gpt-3.5,0.57,1.00,0.61,0.44,0.56,0.58
gpt-4,0.58,0.61,1.00,0.50,0.51,0.61
human_jansen,0.57,0.58,0.61,0.48,0.26,1.00
human_sebhar,0.61,0.44,0.50,1.00,0.55,0.48
llama-2-70b-chat,0.59,0.56,0.51,0.55,1.00,0.26


xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: spearman
xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
METHOD: spearman --average_over_datasets


model,deberta,gpt-3.5,gpt-4,human_sebhar,llama-2-70b-chat,human_jansen
model,,,,,,
deberta,1.00,0.57,0.58,0.61,0.59,0.57
gpt-3.5,0.57,1.00,0.61,0.44,0.56,0.58
gpt-4,0.58,0.61,1.00,0.50,0.51,0.61
human_jansen,0.57,0.58,0.61,0.48,0.26,1.00
human_sebhar,0.61,0.44,0.50,1.00,0.55,0.48
llama-2-70b-chat,0.59,0.56,0.51,0.55,1.00,0.26
